In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
mpl.rcParams['pdf.fonttype'] = 42

In [ ]:
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import defaultdict
import matplotlib.gridspec as gridspec

# Sample configurations
samples = {
    "Visium29_B1": {"fold": 1},
    "Visium29_C1": {"fold": 4},
    "Visium38_B1": {"fold": 2},
    "Visium38_D1": {"fold": 3},
    "Visium37_D1": {"fold": 0}
}

# Base paths
base_path = Path("/QRISdata/Q1851/Xiao/Working_project/benchmarking")

def get_tissue_image_from_adata(adata_gt, sample_name):
    """Extract hires tissue image from the h5ad file"""
    try:
        library_id = f"{sample_name}_hires_image"
        
        if 'spatial' in adata_gt.uns and library_id in adata_gt.uns['spatial']:
            if 'images' in adata_gt.uns['spatial'][library_id]:
                if 'hires' in adata_gt.uns['spatial'][library_id]['images']:
                    img = adata_gt.uns['spatial'][library_id]['images']['hires']
                    print(f"  Loaded hires image from h5ad: shape = {img.shape}")
                    return img, 1.0
        
        print(f"  Could not find hires image in h5ad")
        return None, 1.0
        
    except Exception as e:
        print(f"  Error extracting image: {e}")
        return None, 1.0

def get_method_paths(method_name, sample_name, fold):
    """Get file paths for each method based on their naming conventions"""
    
    if method_name == 'BLEEP':
        pred_file = f"{sample_name}_bleep_virchow2_BC_data_fold_{fold}_predicted.h5ad"
        cor_file = f"bleep_bleep_virchow2_BC_data_{sample_name}_{fold}_clinical_cor.csv"
        mtx_file = None
        folder = "BLEEP"
        
    elif method_name == 'DEEPPT':
        pred_file = f"{sample_name}_deeppt_virchow2_fold_{fold}_predicted.h5ad"
        cor_file = f"deeppt_deeppt_virchow2_{sample_name}_{fold}_clinical_cor.csv"
        mtx_file = None
        folder = "DEEPPT"
        
    elif method_name == 'DeepSpace':
        pred_file = f"{sample_name}_deepspace_fold_{fold}_predicted.h5ad"
        cor_file = f"deepspace_{sample_name}_{fold}_clinical_cor.csv"
        mtx_file = None
        folder = "DeepSpace"
        
    elif method_name == 'STimage':
        pred_file = None
        cor_file = f"stimage_stimage_virchow2_{sample_name}_11_clinical_cor.csv"
        mtx_file = f"stimage_stimage_virchow2_{sample_name}_11_clinical_mtx.csv"
        folder = "STimage"
    
    else:
        raise ValueError(f"Unknown method: {method_name}")
    
    pred_path = base_path / folder / "Skin_data" / pred_file if pred_file else None
    cor_path = base_path / folder / "Skin_data" / cor_file
    mtx_path = base_path / folder / "Skin_data" / mtx_file if mtx_file else None
    
    return pred_path, cor_path, mtx_path

def load_stimage_from_csv(mtx_path, adata_reference):
    """Load STimage prediction from CSV matrix file"""
    try:
        df_mtx = pd.read_csv(mtx_path, index_col=0)
        print(f" Loaded matrix CSV: {df_mtx.shape}")
        
        adata = ad.AnnData(
            X=df_mtx.values,
            obs=adata_reference.obs.copy(),
            var=pd.DataFrame(index=df_mtx.columns),
            obsm={'spatial': adata_reference.obsm['spatial'].copy()}
        )
        
        print(f" Created AnnData from CSV: {adata.shape}")
        return adata
        
    except Exception as e:
        print(f" Error loading STimage from CSV: {e}")
        return None

def load_correlation_file(cor_path, method_name):
    """Load correlation file with method-specific handling"""
    try:
        df_cor = pd.read_csv(cor_path)
        
        if method_name == 'STimage':
            if 'Gene' in df_cor.columns and 'Pearson correlation' in df_cor.columns:
                df_cor = df_cor.set_index('Gene')
                df_cor = df_cor.rename(columns={'Pearson correlation': 'r'})
                df_cor = df_cor[['r']]
                print(f" Converted STimage format to standard format")
            else:
                print(f" Warning: STimage file has unexpected format: {df_cor.columns.tolist()}")
        else:
            if df_cor.columns[0] != 'r':
                df_cor = df_cor.set_index(df_cor.columns[0])
        
        print(f" Loaded correlations: {df_cor.shape}, columns: {df_cor.columns.tolist()}")
        return df_cor
        
    except Exception as e:
        print(f" Error loading correlation file: {e}")
        return None


# STEP 1: Find GLOBAL top 3 genes and best sample for each gene


print("STEP 1: Finding GLOBAL top 3 genes and best sample for each")

# Store correlations per gene across all samples/methods
global_gene_correlations = defaultdict(list)
# Store correlations per gene per sample (average across methods)
gene_sample_correlations = defaultdict(lambda: defaultdict(list))

for sample_name, config in samples.items():
    fold = config['fold']
    print(f"\nCollecting correlations from {sample_name} (fold {fold})...")
    
    for method_name in ['BLEEP', 'DEEPPT', 'DeepSpace', 'STimage']:
        pred_path, cor_path, mtx_path = get_method_paths(method_name, sample_name, fold)
        
        try:
            df_cor = load_correlation_file(cor_path, method_name)
            if df_cor is not None:
                for gene in df_cor.index:
                    pcc = df_cor.loc[gene, 'r']
                    global_gene_correlations[gene].append(pcc)
                    gene_sample_correlations[gene][sample_name].append(pcc)
                print(f"  {method_name}: {len(df_cor)} genes")
        except Exception as e:
            print(f" Error loading {method_name}: {e}")

# Calculate global average PCC for each gene
avg_global_pcc = {}
for gene, corrs in global_gene_correlations.items():
    avg_global_pcc[gene] = np.mean(corrs)

# Get GLOBAL top 3 genes
top_genes_global = sorted(avg_global_pcc.items(), key=lambda x: x[1], reverse=True)[:2]

print("GLOBAL Top 2 genes:")

# For each top gene, find the best sample
top_genes_with_best_sample = []
for i, (gene, avg_pcc) in enumerate(top_genes_global, 1):
    # Calculate average PCC per sample for this gene
    sample_avg_pcc = {}
    for sample_name in samples.keys():
        if sample_name in gene_sample_correlations[gene]:
            sample_avg_pcc[sample_name] = np.mean(gene_sample_correlations[gene][sample_name])
    
    # Find best sample
    if sample_avg_pcc:
        best_sample = max(sample_avg_pcc.items(), key=lambda x: x[1])
        best_sample_name, best_sample_pcc = best_sample
        top_genes_with_best_sample.append({
            'gene': gene,
            'global_avg_pcc': avg_pcc,
            'best_sample': best_sample_name,
            'best_sample_pcc': best_sample_pcc
        })
        print(f"  {i}. {gene:15s} : global avg={avg_pcc:.4f}, best={best_sample_name} (avg={best_sample_pcc:.4f})")


# STEP 2: Plot each gene with its best sample only

print(f"STEP 2: Plotting {len(top_genes_with_best_sample)} genes (1 best sample each)")

n_genes = len(top_genes_with_best_sample)

# Create figure: 3 rows × 6 columns (Tissue, GT, BL, DP, DS, ST)
fig = plt.figure(figsize=(30,5*n_genes))
gs = gridspec.GridSpec(n_genes, 7,  # 6 plot columns + 1 colorbar
                       wspace=0.01, hspace=0.2,
                       width_ratios=[1, 1, 1, 1, 1, 1, 0.05],
                       left=0.05, right=0.95, top=0.93, bottom=0.02)

# Process each gene with its best sample
for gene_idx, gene_info in enumerate(top_genes_with_best_sample):
    gene = gene_info['gene']
    sample_name = gene_info['best_sample']
    fold = samples[sample_name]['fold']
    
    print(f"\n Plotting gene {gene_idx+1}/{n_genes}: {gene} on {sample_name}")
    
    try:
        # Load ground truth
        gt_path = base_path / "Skin_data" / f"{sample_name}_virchow2.h5ad"
        adata_gt = ad.read_h5ad(str(gt_path))
        
        # Get tissue image
        tissue_img, _ = get_tissue_image_from_adata(adata_gt, sample_name)
        if tissue_img is None:
            continue
        
        # Load predictions
        method_predictions = {}
        method_correlations = {}
        
        for method_name in ['BLEEP', 'DEEPPT', 'DeepSpace', 'STimage']:
            pred_path, cor_path, mtx_path = get_method_paths(method_name, sample_name, fold)
            
            try:
                if method_name == 'STimage':
                    if mtx_path and mtx_path.exists():
                        adata_pred = load_stimage_from_csv(mtx_path, adata_gt)
                    else:
                        adata_pred = None
                else:
                    if pred_path and pred_path.exists():
                        adata_pred = ad.read_h5ad(str(pred_path))
                    else:
                        adata_pred = None
                
                if adata_pred is not None:
                    df_cor = load_correlation_file(cor_path, method_name)
                    if df_cor is not None:
                        method_predictions[method_name] = adata_pred
                        method_correlations[method_name] = df_cor
            except:
                pass
        
        # Check if gene exists
        if gene not in adata_gt.var_names:
            print(f"  Gene {gene} not found in {sample_name}")
            continue
        
        # Get gene expression
        gt_exp = adata_gt[:, gene].X
        if hasattr(gt_exp, 'toarray'):
            gt_exp = gt_exp.toarray().flatten()
        else:
            gt_exp = gt_exp.flatten()
        
        coords = adata_gt.obsm['spatial']
        
        # Calculate color scale for GT
        gt_vmin, gt_vmax = np.percentile(gt_exp[gt_exp > 0], [2, 98]) if (gt_exp > 0).sum() > 0 else (gt_exp.min(), gt_exp.max())
        
        # 1. Tissue
        ax = fig.add_subplot(gs[gene_idx, 0])
        ax.imshow(tissue_img)
        ax.set_title(f'{sample_name}', fontsize=30, fontweight='bold', pad=4)
        ax.set_xlim(0, tissue_img.shape[1])
        ax.set_ylim(tissue_img.shape[0], 0)
        ax.axis('off')
        
        # 2. Ground Truth
        ax = fig.add_subplot(gs[gene_idx, 1])
        sc = ax.scatter(coords[:, 0], coords[:, 1], 
                      c=gt_exp, cmap='viridis', 
                      s=18, vmin=gt_vmin, vmax=gt_vmax,
                      edgecolors='white', linewidths=0.1, alpha=0.95)
        ax.set_title('Ground Truth', fontsize=30, fontweight='bold', pad=4)
        ax.set_xlim(0, tissue_img.shape[1])
        ax.set_ylim(tissue_img.shape[0], 0)
        ax.set_facecolor('white')
        ax.axis('off')
        ax.set_aspect('equal')
        
        # 3-6. Methods
        method_names = ['BLEEP', 'DEEPPT', 'DeepSpace', 'STimage']
        method_short = ['BLEEP', 'DEEPPT', 'DeepSpace', 'STimage']
        
        for method_idx, (method_name, short_name) in enumerate(zip(method_names, method_short)):
            ax = fig.add_subplot(gs[gene_idx, 2 + method_idx])
            
            if method_name in method_predictions and gene in method_predictions[method_name].var_names:
                pred_exp = method_predictions[method_name][:, gene].X
                if hasattr(pred_exp, 'toarray'):
                    pred_exp = pred_exp.toarray().flatten()
                else:
                    pred_exp = pred_exp.flatten()
                
                # Individual scale
                if (pred_exp > 0).sum() > 0:
                    pred_vmin, pred_vmax = np.percentile(pred_exp[pred_exp > 0], [2, 98])
                else:
                    pred_vmin, pred_vmax = pred_exp.min(), pred_exp.max()
                
                if pred_vmin == pred_vmax:
                    pred_vmin, pred_vmax = pred_exp.min(), pred_exp.max()
                
                ax.scatter(coords[:, 0], coords[:, 1], 
                         c=pred_exp, cmap='viridis', 
                         s=18, vmin=pred_vmin, vmax=pred_vmax,
                         edgecolors='white', linewidths=0.1, alpha=0.95)
                
                # Add PCC value
                if method_name in method_correlations and gene in method_correlations[method_name].index:
                    pcc = method_correlations[method_name].loc[gene, 'r']
                    ax.set_title(f'{short_name}\nPCC: {pcc:.3f}', fontsize=30, fontweight='bold', pad=4)
                else:
                    ax.set_title(short_name, fontsize=30, fontweight='bold', pad=4)
            else:
                ax.text(0.5, 0.5, 'N/A', ha='center', va='center', 
                       transform=ax.transAxes, fontsize=30)
                ax.set_title(short_name, fontsize=30, fontweight='bold', pad=4)
            
            ax.set_xlim(0, tissue_img.shape[1])
            ax.set_ylim(tissue_img.shape[0], 0)
            ax.set_facecolor('white')
            ax.axis('off')
            ax.set_aspect('equal')
        
        # Add gene name as row label (left side)
        fig.text(0.01, 0.93 - (gene_idx + 0.5) * (0.88 / n_genes), 
                 f'{gene}',
                 fontsize=30, fontweight='bold', rotation=0, 
                 va='center', ha='left')
    
    except Exception as e:
        print(f"  Error: {e}")
        import traceback
        traceback.print_exc()
        continue

# Add colorbar
cbar_ax = fig.add_subplot(gs[:, -1])
cbar = plt.colorbar(sc, cax=cbar_ax, orientation='vertical')
cbar.set_label('', fontsize=10, fontweight='bold')
cbar.ax.tick_params(labelsize=8)


# Save

plt.savefig('d2.png', dpi=600, bbox_inches='tight')
plt.savefig('d2.pdf', dpi=600, bbox_inches='tight')
plt.close()

print("Complete!")
print(f"Summary:")
print(f"  - Found global top 3 genes across all samples")
print(f"  - For each gene, selected the best performing sample")
print(f"  - Plotted 3 rows (genes) × 1 sample each")
print(f"  - Output: 'd2.png' and 'd2.pdf'")


In [ ]:
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Sample configurations for BC_data
samples = {
    "A1": {"fold": 0}, "A2": {"fold": 1}, "A3": {"fold": 2}, "A4": {"fold": 3}, "A5": {"fold": 4}, "A6": {"fold": 5},
    "B1": {"fold": 6}, "B2": {"fold": 7}, "B3": {"fold": 8}, "B4": {"fold": 9}, "B5": {"fold": 10}, "B6": {"fold": 11},
    "C1": {"fold": 12}, "C2": {"fold": 13}, "C3": {"fold": 14}, "C4": {"fold": 15}, "C5": {"fold": 16}, "C6": {"fold": 17},
    "D1": {"fold": 18}, "D2": {"fold": 19}, "D3": {"fold": 20}, "D4": {"fold": 21}, "D5": {"fold": 22}, "D6": {"fold": 23},
    "E1": {"fold": 24}, "E2": {"fold": 25}, "E3": {"fold": 26},
    "F1": {"fold": 27}, "F2": {"fold": 28}, "F3": {"fold": 29},
    "G1": {"fold": 30}, "G2": {"fold": 31}, "G3": {"fold": 32},
    "H1": {"fold": 33}, "H2": {"fold": 34}, "H3": {"fold": 35}
}

base_path = Path("/QRISdata/Q1851/Xiao/Working_project/benchmarking")

def get_tissue_image_from_adata(adata_gt, sample_name):
    try:
        library_id = sample_name
        if 'spatial' in adata_gt.uns and library_id in adata_gt.uns['spatial']:
            if 'images' in adata_gt.uns['spatial'][library_id]:
                if 'fulres' in adata_gt.uns['spatial'][library_id]['images']:
                    return adata_gt.uns['spatial'][library_id]['images']['fulres'], 1.0
                elif 'hires' in adata_gt.uns['spatial'][library_id]['images']:
                    return adata_gt.uns['spatial'][library_id]['images']['hires'], 1.0
        return None, 1.0
    except:
        return None, 1.0

def filter_adata_by_sample(adata, sample_name):
    return adata[adata.obs['library_id'] == sample_name].copy()

def get_method_paths(method_name, sample_name, fold):
    if method_name == 'BLEEP':
        pred_file = f"{sample_name}_bleep_virchow2_BC_data_fold_{fold}_predicted.h5ad"
        cor_file = f"bleep_bleep_virchow2_BC_data_{sample_name}_{fold}_clinical_cor.csv"
        mtx_file = None
        folder = "BLEEP"
    elif method_name == 'DEEPPT':
        pred_file = f"{sample_name}_deeppt_virchow2_fold_{fold}_predicted.h5ad"
        cor_file = f"deeppt_deeppt_virchow2_{sample_name}_{fold}_clinical_cor.csv"
        mtx_file = None
        folder = "DEEPPT"
    elif method_name == 'DeepSpace':
        pred_file = f"{sample_name}_deepspace_fold_{fold}_predicted.h5ad"
        cor_file = f"deepspace_{sample_name}_{fold}_clinical_cor.csv"
        mtx_file = None
        folder = "DeepSpace"
    elif method_name == 'STimage':
        pred_file = None
        cor_file = f"stimage_stimage_virchow2_{sample_name}_11_clinical_cor.csv"
        mtx_file = f"stimage_stimage_virchow2_{sample_name}_11_clinical_mtx.csv"
        folder = "STimage"
    else:
        raise ValueError(f"Unknown method: {method_name}")
    
    pred_path = base_path / folder / "BC_data" / pred_file if pred_file else None
    cor_path = base_path / folder / "BC_data" / cor_file
    mtx_path = base_path / folder / "BC_data" / mtx_file if mtx_file else None
    return pred_path, cor_path, mtx_path

def load_stimage_from_csv(mtx_path, adata_reference):
    try:
        df_mtx = pd.read_csv(mtx_path, index_col=0)
        obs_subset = adata_reference.obs.iloc[:df_mtx.shape[0]].copy()
        spatial_subset = adata_reference.obsm['spatial'][:df_mtx.shape[0]].copy()
        return ad.AnnData(X=df_mtx.values, obs=obs_subset, 
                         var=pd.DataFrame(index=df_mtx.columns), 
                         obsm={'spatial': spatial_subset})
    except:
        return None

def load_correlation_file(cor_path, method_name):
    try:
        df_cor = pd.read_csv(cor_path)
        if method_name == 'STimage':
            if 'Gene' in df_cor.columns and 'Pearson correlation' in df_cor.columns:
                df_cor = df_cor.set_index('Gene')
                df_cor = df_cor.rename(columns={'Pearson correlation': 'r'})
                df_cor = df_cor[['r']]
        else:
            if df_cor.columns[0] != 'r':
                df_cor = df_cor.set_index(df_cor.columns[0])
        return df_cor
    except:
        return None

def plot_combined_best_genes_samples(gene_sample_pairs, all_data):
    n_genes = len(gene_sample_pairs)
    fig = plt.figure(figsize=(30,5*n_genes))
    gs = fig.add_gridspec(n_genes, 7,  # 6 plot columns + 1 colorbar
                       wspace=0.05, hspace=0.2,
                       width_ratios=[1, 1, 1, 1, 1, 1, 0.05],
                       left=0.05, right=0.95, top=0.93, bottom=0.02)
    method_names = ['BLEEP', 'DEEPPT', 'DeepSpace', 'STimage']
    
    # Collect all expressions for shared color scale
    all_expressions = []
    for gene, sample_name in gene_sample_pairs:
        sample_data = all_data[sample_name]
        adata_gt = sample_data['adata_gt']
        method_predictions = sample_data['method_predictions']
        
        if gene in adata_gt.var_names:
            gt_exp = adata_gt[:, gene].X
            gt_exp = gt_exp.toarray().flatten() if hasattr(gt_exp, 'toarray') else gt_exp.flatten()
            all_expressions.append(gt_exp)
            
            for adata_pred in method_predictions.values():
                if adata_pred is not None and gene in adata_pred.var_names:
                    pred_exp = adata_pred[:, gene].X
                    pred_exp = pred_exp.toarray().flatten() if hasattr(pred_exp, 'toarray') else pred_exp.flatten()
                    all_expressions.append(pred_exp)
    
    all_exp_concat = np.concatenate(all_expressions) if len(all_expressions) > 0 else np.array([0, 1])
    vmin, vmax = np.percentile(all_exp_concat, 2), np.percentile(all_exp_concat, 98)
    
    # Track previous sample to know when to add sample name
    prev_sample = None
    
    # Plot each gene-sample pair
    for row_idx, (gene, sample_name) in enumerate(gene_sample_pairs):
        sample_data = all_data[sample_name]
        tissue_img = sample_data['tissue_img']
        adata_gt = sample_data['adata_gt']
        method_predictions = sample_data['method_predictions']
        method_correlations = sample_data['method_correlations']
        
        coords = None
        for adata_pred in method_predictions.values():
            if adata_pred is not None:
                coords = adata_pred.obsm['spatial']
                break
        if coords is None:
            coords = adata_gt.obsm['spatial']
        
        if gene not in adata_gt.var_names:
            continue
        
        gt_exp = adata_gt[:, gene].X
        gt_exp = gt_exp.toarray().flatten() if hasattr(gt_exp, 'toarray') else gt_exp.flatten()
        
        # Check if this is a new sample (first row or sample changed)
        is_new_sample = (prev_sample is None or prev_sample != sample_name)
        prev_sample = sample_name
        
        # Tissue
        ax_tissue = fig.add_subplot(gs[row_idx, 0])
        ax_tissue.imshow(tissue_img)
        ax_tissue.set_xlim(0, tissue_img.shape[1])
        ax_tissue.set_ylim(tissue_img.shape[0], 0)
        ax_tissue.axis('off')
        
        # Add "Visium38_[sample]" title only for first row of each sample
        ax_tissue.set_title(f'Visium38_{sample_name}', fontsize=30, fontweight='bold', pad=10)
        
        # Gene name on the left
        ax_tissue.text(-0.12, 0.5, gene, transform=ax_tissue.transAxes,
                      fontsize=30, fontweight='bold', va='center', ha='right')
        
        # Ground Truth
        ax_gt = fig.add_subplot(gs[row_idx, 1])
        sc = ax_gt.scatter(coords[:, 0], coords[:, 1], c=gt_exp, cmap='viridis',
                          s=100, vmin=vmin, vmax=vmax, edgecolors='none', alpha=0.9)
        ax_gt.set_xlim(0, tissue_img.shape[1])
        ax_gt.set_ylim(tissue_img.shape[0], 0)
        ax_gt.set_facecolor('white')
        ax_gt.axis('off')
        ax_gt.set_aspect('equal')
        
        # Add "Ground Truth" title only for first row of each sample
        
        ax_gt.set_title('Ground Truth', fontsize=30, fontweight='bold', pad=10)
        
        # Methods - ALWAYS show method name and PCC for ALL rows
        for col_idx, method_name in enumerate(method_names):
            ax = fig.add_subplot(gs[row_idx, col_idx + 2])
            
            if method_name in method_predictions and method_predictions[method_name] is not None:
                adata_pred = method_predictions[method_name]
                if gene in adata_pred.var_names:
                    pred_exp = adata_pred[:, gene].X
                    pred_exp = pred_exp.toarray().flatten() if hasattr(pred_exp, 'toarray') else pred_exp.flatten()
                    ax.scatter(coords[:, 0], coords[:, 1], c=pred_exp, cmap='viridis',
                              s=100, vmin=vmin, vmax=vmax, edgecolors='none', alpha=0.9)
                    
                    # Get correlation
                    corr = method_correlations.get(method_name, {}).get(gene, np.nan)
                    
                    # ALWAYS show method name and PCC (for all rows)
                    ax.set_title(f'{method_name}\nPCC: {corr:.3f}', 
                               fontsize=30, fontweight='bold', pad=5)
                else:
                    ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes, fontsize=30)
                    if is_new_sample:
                        ax.set_title(f'{method_name}\nN/A', fontsize=30, fontweight='bold', pad=5)
            else:
                ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes, fontsize=30)
                if is_new_sample:
                    ax.set_title(f'{method_name}\nN/A', fontsize=30, fontweight='bold', pad=5)
            
            ax.set_xlim(0, tissue_img.shape[1])
            ax.set_ylim(tissue_img.shape[0], 0)
            ax.set_facecolor('white')
            ax.axis('off')
            ax.set_aspect('equal')
    
    # Colorbar
    cbar_ax = fig.add_subplot(gs[:, -1])
    cbar = plt.colorbar(sc, cax=cbar_ax, orientation='vertical')
    # cbar.set_label('', fontsize=10, fontweight='bold')
    cbar.ax.tick_params(labelsize=8)
    
    # Simple or no main title (to match the reference style)
    # fig.suptitle('Top 3 Genes with Best Samples', fontsize=30, fontweight='bold', y=0.98)
    
    return fig



# STEP 1: Find samples with ALL 4 methods


samples_with_all_methods = []

for sample_name in samples.keys():
    fold = samples[sample_name]['fold']
    methods_available = []
    
    for method_name in ['BLEEP', 'DEEPPT', 'DeepSpace', 'STimage']:
        pred_path, cor_path, mtx_path = get_method_paths(method_name, sample_name, fold)
        
        if method_name == 'STimage':
            if mtx_path and mtx_path.exists() and cor_path.exists():
                methods_available.append(method_name)
        else:
            if pred_path and pred_path.exists() and cor_path.exists():
                methods_available.append(method_name)
    
    if len(methods_available) == 4:
        samples_with_all_methods.append(sample_name)

print(f" Total {len(samples_with_all_methods)} samples: {', '.join(samples_with_all_methods)}")

if len(samples_with_all_methods) == 0:
    print("No sample")
    exit()


# STEP 2: In those samples, find genes present in ALL 4 methods



gene_sample_method_data = {}

for sample_name in samples_with_all_methods:
    fold = samples[sample_name]['fold']
    method_genes = {}
    
    for method_name in ['BLEEP', 'DEEPPT', 'DeepSpace', 'STimage']:
        _, cor_path, _ = get_method_paths(method_name, sample_name, fold)
        df_cor = load_correlation_file(cor_path, method_name)
        if df_cor is not None:
            method_genes[method_name] = set(df_cor.index)
            for gene in df_cor.index:
                if gene not in gene_sample_method_data:
                    gene_sample_method_data[gene] = {}
                if sample_name not in gene_sample_method_data[gene]:
                    gene_sample_method_data[gene][sample_name] = {}
                gene_sample_method_data[gene][sample_name][method_name] = df_cor.loc[gene, 'r']
    
    if len(method_genes) == 4:
        common = set.intersection(*method_genes.values())
        print(f"  {sample_name}: {len(common)} common genes ")

# Find genes with all 4 methods on multiple samples
genes_with_full_coverage = {}
for gene, sample_data in gene_sample_method_data.items():
    samples_with_4_methods = sum(1 for m in sample_data.values() if len(m) == 4)
    if samples_with_4_methods >= 3:  # At least 3 samples
        all_pccs = [pcc for sample_methods in sample_data.values() for pcc in sample_methods.values()]
        genes_with_full_coverage[gene] = np.mean(all_pccs)

sorted_genes = sorted(genes_with_full_coverage.items(), key=lambda x: x[1], reverse=True)
print(f"\n {len(sorted_genes)} genes with full coverage on at least 3 samples.")
print("\nTop 10:")
for i, (gene, pcc) in enumerate(sorted_genes[:10], 1):
    print(f"  {i}. {gene:15s}: {pcc:.4f}")

top_3_genes = [g[0] for g in sorted_genes[:2]]
print(f"\n Top 3: {', '.join(top_3_genes)}")



# STEP 3: Find best sample for each gene


gene_best_sample = []
for gene in top_3_genes:
    samples_4_methods = []
    for sample_name, method_data in gene_sample_method_data[gene].items():
        if len(method_data) == 4:
            samples_4_methods.append((sample_name, np.mean(list(method_data.values()))))
    
    if samples_4_methods:
        samples_4_methods.sort(key=lambda x: x[1], reverse=True)
        best = samples_4_methods[0]
        gene_best_sample.append((gene, best[0]))
        print(f"  {gene:15s} → {best[0]} (PCC: {best[1]:.4f})")


# STEP 4: Load and plot


gt_path = base_path / "BC_data" / "all_adata_img.h5ad"
adata_gt_full = ad.read_h5ad(str(gt_path))

unique_samples = list(set([s for _, s in gene_best_sample]))
all_data = {}

for sample_name in unique_samples:
    fold = samples[sample_name]['fold']
    adata_gt_sample = filter_adata_by_sample(adata_gt_full, sample_name)
    tissue_img, _ = get_tissue_image_from_adata(adata_gt_full, sample_name)
    
    if tissue_img is None:
        continue
    
    method_predictions = {}
    method_correlations = {}
    
    for method_name in ['BLEEP', 'DEEPPT', 'DeepSpace', 'STimage']:
        pred_path, cor_path, mtx_path = get_method_paths(method_name, sample_name, fold)
        
        if method_name == 'STimage':
            adata_pred = load_stimage_from_csv(mtx_path, adata_gt_sample) if mtx_path.exists() else None
        else:
            adata_pred = ad.read_h5ad(str(pred_path)) if pred_path.exists() else None
        
        if adata_pred is not None:
            df_cor = load_correlation_file(cor_path, method_name)
            if df_cor is not None:
                method_predictions[method_name] = adata_pred
                method_correlations[method_name] = {g: df_cor.loc[g, 'r'] for g in df_cor.index}
    
    genes_needed = [g for g, s in gene_best_sample if s == sample_name]
    adata_gt_sub = adata_gt_sample[:, genes_needed].copy()
    for m in method_predictions:
        method_predictions[m] = method_predictions[m][:, genes_needed].copy()
    
    all_data[sample_name] = {
        'tissue_img': tissue_img,
        'adata_gt': adata_gt_sub,
        'method_predictions': method_predictions,
        'method_correlations': method_correlations
    }

fig = plot_combined_best_genes_samples(gene_best_sample, all_data)
# output_dir = Path("spatial_plots_comparison/BC_data/final")
# output_dir.mkdir(parents=True, exist_ok=True)
# output_file = "d1.png"
fig.savefig("d1.png", dpi=600, bbox_inches='tight')
fig.savefig("d1.pdf", dpi=600, bbox_inches='tight')
plt.close(fig)

